# En este ipynb se muestra la creación de la ontología usada de animes.

In [2]:
import sbc_tools as sbc
from SPARQLWrapper import SPARQLWrapper, JSON
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL

In [3]:
grafo = Graph()
grafo = sbc.load("anime_knowledge_graph_clean.ttl")

sbc.print_class_hierarchy(grafo)


JERARQUÍA DE CLASES DE LA ONTOLOGÍA
📁 Production Role (anime-ontology#ProductionRole) [anime-ontology#productionRoleIn]
  └─ 2nd_Key_Animation: Sin etiqueta
  └─ ADR_Director: Sin etiqueta
  └─ Animation_Check: Sin etiqueta
  └─ Animation_Director: Sin etiqueta
  └─ Art_Director: Sin etiqueta
  └─ Assistant_Animation_Director: Sin etiqueta
  └─ Assistant_Director: Sin etiqueta
  └─ Assistant_Producer: Sin etiqueta
  └─ Associate_Producer: Sin etiqueta
  └─ Background_Art: Sin etiqueta
  └─ Casting_Director: Sin etiqueta
  └─ Character_Design: Sin etiqueta
  └─ Chief_Animation_Director: Sin etiqueta
  └─ CoDirector: Sin etiqueta
  └─ CoProducer: Sin etiqueta
  └─ Color_Design: Sin etiqueta
  └─ Color_Setting: Sin etiqueta
  └─ Creator: Sin etiqueta
  └─ Dialogue_Editing: Sin etiqueta
  └─ Digital_Paint: Sin etiqueta
  └─ Director: Sin etiqueta
  └─ Director_of_Photography: Sin etiqueta
  └─ Editing: Sin etiqueta
  └─ Episode_Director: Sin etiqueta
  └─ Executive_Producer: Sin etiqueta


In [5]:
from rdflib import Graph, Namespace, Literal, RDF, XSD, URIRef
import requests
import re
import time

def clean_uri(name):
    name = name.strip()
    name = re.sub(r'\s+', '_', name)
    name = re.sub(r'[^a-zA-Z0-9_]', '', name)
    return name

max_consecutive_errors = 50
errors = 0

#empezamos el time
start = time.time()

# Cargar grafo existente
g = sbc.load("ontologiaAnimesIncompleta.ttl", format="turtle")

# Namespaces
ONTO = Namespace("http://www.fdi.ucm.es/mia/sbc/animeontology/")
NS2 = Namespace("http://www.semanticweb.org/eladi/ontologies/anime-ontology#")
anime_id = 1
while True:
    
    try:
        anime_data = requests.get(f"https://api.jikan.moe/v4/anime/{anime_id}").json()['data']
    except:
        print(f"Anime ID {anime_id} error")
        errors += 1
        anime_id += 1

        if errors >= max_consecutive_errors:
            print("Probablemente no quedan más animes válidos. Deteniendo.")
            break

        continue



    if errors >= max_consecutive_errors:
        print("Probablemente no quedan más animes válidos. Deteniendo.")
        break


    errors = 0 
    title = anime_data.get('title_english') or anime_data['title']
    anime_uri = ONTO[clean_uri(title)]

    # AnimeSeries
    if (anime_uri, RDF.type, ONTO.AnimeSeries) not in g:
        g.add((anime_uri, RDF.type, ONTO.AnimeSeries))

    # Títulos
    for key, value in [('englishTitle', anime_data.get('title_english')), ('episodeTitle', anime_data['title'])]:
        if value:
            pred = getattr(NS2, key)
            lit = Literal(value, datatype=XSD.string)
            if (anime_uri, pred, lit) not in g:
                g.add((anime_uri, pred, lit))

    # Número de episodios y fecha
    if anime_data.get('episodes'):
        lit = Literal(anime_data['episodes'], datatype=XSD.integer)
        if (anime_uri, ONTO.numEpisodes, lit) not in g:
            g.add((anime_uri, ONTO.numEpisodes, lit))
    if anime_data.get('aired') and anime_data['aired'].get('from'):
        lit = Literal(anime_data['aired']['from'], datatype=XSD.dateTime)
        if (anime_uri, ONTO.startDate, lit) not in g:
            g.add((anime_uri, ONTO.startDate, lit))

    # Géneros
    for genre in anime_data.get('genres', []):
        genre_uri = ONTO[clean_uri(genre['name'])]
        if (genre_uri, RDF.type, ONTO.Genre) not in g:
            g.add((genre_uri, RDF.type, ONTO.Genre))
            g.add((genre_uri, NS2.label, Literal(genre['name'], datatype=XSD.string)))
        if (anime_uri, ONTO.hasGenre, genre_uri) not in g:
            g.add((anime_uri, ONTO.hasGenre, genre_uri))

    # Estudios
    for studio in anime_data.get('studios', []):
        studio_uri = ONTO[clean_uri(studio['name'])]
        if (studio_uri, RDF.type, NS2.Studio) not in g:
            g.add((studio_uri, RDF.type, NS2.Studio))
            g.add((studio_uri, NS2.label, Literal(studio['name'], datatype=XSD.string)))
        if (studio_uri, NS2.produced, anime_uri) not in g:
            g.add((studio_uri, NS2.produced, anime_uri))

    # === Staff (directores, productores, etc.) ===
    try:
        staff_data = requests.get(f"https://api.jikan.moe/v4/anime/{anime_id}/staff").json()['data']
        for s in staff_data:
            staff_uri = ONTO[clean_uri(s['person']['name'])]
            if (staff_uri, RDF.type, NS2.RealPerson) not in g:
                g.add((staff_uri, RDF.type, NS2.RealPerson))
                g.add((staff_uri, NS2.label, Literal(s['person']['name'], datatype=XSD.string)))
            # Cada posición como rol de producción
            for pos in s.get('positions', []):
                role_uri = ONTO[clean_uri(pos)]
                if (role_uri, RDF.type, NS2.ProductionRole) not in g:
                    g.add((role_uri, RDF.type, NS2.ProductionRole))
                    g.add((role_uri, NS2.label, Literal(pos, datatype=XSD.string)))
                if (role_uri, NS2.roleIn, anime_uri) not in g:
                    g.add((role_uri, NS2.roleIn, anime_uri))
    except:
        pass

    # === Characters y VoiceActors ===
    try:
        char_data = requests.get(f"https://api.jikan.moe/v4/anime/{anime_id}/characters").json()['data']
        for c in char_data:
            char_uri = ONTO[clean_uri(c['character']['name'])]
            if (char_uri, RDF.type, NS2.Character) not in g:
                g.add((char_uri, RDF.type, NS2.Character))
                g.add((char_uri, NS2.label, Literal(c['character']['name'], datatype=XSD.string)))
            if (c['role'].lower() == 'main'):
                if (char_uri, RDF.type, NS2.MainCharacter) not in g:
                    g.add((char_uri, RDF.type, NS2.MainCharacter))
            if (char_uri, NS2.characterIn, anime_uri) not in g:
                g.add((char_uri, NS2.characterIn, anime_uri))
            # Voice actors
            for va in c.get('voice_actors', []):
                if va['language'].lower() != 'japanese':
                    continue
                va_uri = ONTO[clean_uri(va['person']['name'])]
                if (va_uri, RDF.type, NS2.VoiceActor) not in g:
                    g.add((va_uri, RDF.type, NS2.VoiceActor))
                    g.add((va_uri, NS2.label, Literal(va['person']['name'], datatype=XSD.string)))
                if (char_uri, NS2.dubbedBy, va_uri) not in g:
                    g.add((char_uri, NS2.dubbedBy, va_uri))
    except:
        pass

    # === Episodios ===
    try:
        eps_data = requests.get(f"https://api.jikan.moe/v4/anime/{anime_id}/episodes").json()['data']
        for ep in eps_data:
            ep_uri = ONTO[clean_uri(ep['title'])]
            if (ep_uri, RDF.type, NS2.Movie) not in g:
                g.add((ep_uri, RDF.type, NS2.Episode))  # o ns2:Episode según tu ontología
            if (ep_uri, NS2.episodeTitle, Literal(ep['title'], datatype=XSD.string)) not in g:
                g.add((ep_uri, NS2.episodeTitle, Literal(ep['title'], datatype=XSD.string)))
            if (ep_uri, NS2.isEpisodeOf, anime_uri) not in g:
                g.add((ep_uri, NS2.isEpisodeOf, anime_uri))
    except:
        pass

    print(f"Procesado: {title} (ID {anime_id})")
    time.sleep(0.5)

    anime_id += 1

  

# Guardar grafo actualizado
end = time.time()
print(f"Tiempo total de ejecución: {end - start} segundos")
g.serialize("expansionDeDominioInfinito.ttl", format="turtle")
print("¡Grafo completamente enriquecido listo!")


Procesado: Cowboy Bebop (ID 1)
Anime ID 2 error
Anime ID 3 error
Anime ID 4 error
Procesado: Cowboy Bebop: The Movie (ID 5)
Procesado: Trigun (ID 6)
Procesado: Witch Hunter Robin (ID 7)
Procesado: Beet the Vandel Buster (ID 8)
Anime ID 9 error
Anime ID 10 error
Anime ID 11 error
Anime ID 12 error
Anime ID 13 error
Anime ID 14 error
Procesado: Eyeshield 21 (ID 15)
Procesado: Honey and Clover (ID 16)
Procesado: Hungry Heart: Wild Striker (ID 17)
Procesado: Initial D Fourth Stage (ID 18)
Procesado: Monster (ID 19)
Procesado: Naruto (ID 20)
Procesado: One Piece (ID 21)
Procesado: The Prince of Tennis (ID 22)
Procesado: Ring ni Kakero 1 (ID 23)
Procesado: School Rumble (ID 24)
Procesado: Desert Punk (ID 25)
Procesado: Texhnolyze (ID 26)
Procesado: Trinity Blood (ID 27)
Procesado: Yakitate!! Japan (ID 28)
Procesado: Zipang (ID 29)
Procesado: Neon Genesis Evangelion (ID 30)
Procesado: Neon Genesis Evangelion: Death & Rebirth (ID 31)
Procesado: Neon Genesis Evangelion: The End of Evangelion (I

In [9]:
g.parse("expansionDeDominioInfinito.ttl", format="turtle")
sbc.print_class_hierarchy(g)


JERARQUÍA DE CLASES DE LA ONTOLOGÍA
📁 Production Role (anime-ontology#ProductionRole) [anime-ontology#productionRoleIn]
  └─ 2nd_Key_Animation: Sin etiqueta
  └─ ADR_Director: Sin etiqueta
  └─ Animation_Check: Sin etiqueta
  └─ Animation_Director: Sin etiqueta
  └─ Art_Director: Sin etiqueta
  └─ Assistant_Animation_Director: Sin etiqueta
  └─ Assistant_Director: Sin etiqueta
  └─ Assistant_Producer: Sin etiqueta
  └─ Associate_Producer: Sin etiqueta
  └─ Background_Art: Sin etiqueta
  └─ Casting_Director: Sin etiqueta
  └─ Character_Design: Sin etiqueta
  └─ Chief_Animation_Director: Sin etiqueta
  └─ CoDirector: Sin etiqueta
  └─ CoProducer: Sin etiqueta
  └─ Color_Design: Sin etiqueta
  └─ Color_Setting: Sin etiqueta
  └─ Creator: Sin etiqueta
  └─ Dialogue_Editing: Sin etiqueta
  └─ Digital_Paint: Sin etiqueta
  └─ Director: Sin etiqueta
  └─ Director_of_Photography: Sin etiqueta
  └─ Editing: Sin etiqueta
  └─ Episode_Director: Sin etiqueta
  └─ Executive_Producer: Sin etiqueta


In [10]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL, Literal

# 1. Configuración del Grafo
ONTO = Namespace("http://www.fdi.ucm.es/mia/sbc/animeontology/")
g = Graph()
g.bind("onto", ONTO)
g.bind("rdfs", RDFS)
g.bind("owl", OWL)

# 2. Lista de URIs (La misma que tenías)
genres_uris = [
    'ASMR', 'Monster_of_the_Week', 'Sentai', 'Westerns_on_television', 'Xianxia',
    'action_comics', 'action_manga', 'adventure_manga', 'adventure_video_game',
    'alternate_history', 'alternate_history_television_series',
    'animal-themed_television_series', 'animated_cartoon', 'anime-influenced_animation',
    'anime/manga_style', 'anthology_series', 'apocalyptic_fiction', 'biographical_film',
    'bishōjo_game', 'chuanyue', 'comedy_drama_anime', 'comedy_horror',
    'coming-of-age_fiction', 'coming-of-age_television_program', 'contemporary_fantasy',
    'crime_television_series', 'cringe_comedy', 'documentary_film', 'dystopian_fiction',
    'educational_animation', 'educational_broadcast_program', 'fantastique',
    'fantasy_comics', 'fantasy_television_program', 'fashion', 'film_based_on_literature',
    'fishing', 'food_culture', 'gag_manga', 'gambling', 'gender_bender', "girls'_anime",
    'gods_and_demons_fiction', 'historical_manga', 'historical_mystery', 'horror_film',
    'horror_television_series', 'horror_video_game', 'legal_drama', 'life_simulation_game',
    'light_novel', 'love_comedy', 'mahjong_manga', 'manzai', 'martial_art',
    'massively_multiplayer_online_role-playing_game', 'maze_video_game', 'military_fiction',
    'music', 'music_program', 'music_television', 'mystery_television_series',
    'national_personification', 'parody', 'police_procedural', 'post-apocalyptic_film',
    'progression_fantasy', 'rhythm_game', 'role-playing_video_game',
    'romance_television_series', 'science_fiction_comics', 'science_fiction_film',
    'social_network_game', 'social_simulation_game', 'soundtrack', 'supernatural_fiction',
    'sushi', 'swashbuckler_film', 'tabletop_role-playing_game',
    'tactical_role-playing_game', 'teen_film', 'time-travel_fiction', 'time_travel',
    'tragedy', 'werewolf_anime_and_manga', 'zombie_television_program', 'Japanese_idol',
    'Jidaigeki', 'LGBT-themed_video_game', 'action-adventure_game', 'anime',
    'black_comedy', 'dating_sim', 'documentary', 'drama_film', 'educational_television',
    'girls_with_guns', 'historical_television_series', 'music_video_game', 'portal_fantasy',
    'psychological_horror', 'psychological_thriller', 'romantic_comedy', 'samurai_cinema',
    'science_fiction_video_game', 'sex_comedy', 'supernatural_thriller', 'tech_noir',
    'tokusatsu', 'trial_film', 'variety_show', 'work_life', 'wuxia',
    'zombie_anime_and_manga', "Boys'_Love_game", 'action_game', 'action_role-playing_game',
    'action_television_series', 'adventure_television_series', 'anime_television_program',
    'association_football_video_game', 'band_anime_and_manga', 'comedy', 'comedy_drama',
    'comedy_television_program', 'dark_fantasy_anime_and_manga', 'film_based_on_a_novel',
    'hentai', 'idol_anime_and_manga', 'mystery_fiction', 'occult_detective_fiction',
    'pirate_fiction', 'romance_adventure_game', 'science_fiction',
    'superhero_television_program', 'sword_and_sorcery', 'comedy_film', 'crossover_fiction',
    'detective_fiction', 'fantasy', 'soft_science_fiction', 'space_Western', 'sport_film',
    'BL', 'fantasy_television_series', 'martial_arts_anime', 'school_anime',
    'science_fiction_television_program', 'survival_anime_and_manga',
    'time_travel_in_fiction', 'biopunk', 'drama_anime', 'high_fantasy', 'neo-noir',
    'space_opera', 'adaptation', 'sports_manga', 'drama_television_series', 'steampunk',
    'psychological_anime_and_manga', 'science_fantasy', 'slice_of_life',
    "children's_television_series", 'military_science_fiction', 'suspense_anime_and_manga',
    'thriller_anime_and_manga', 'eroge', 'cyberpunk', 'fantasy_anime',
    'time-travel_television_program', 'iyashikei', 'time-travel_anime_and_manga',
    'adventure_anime', 'anti-war_fiction', 'action_anime', 'yaoi',
    'comedy_television_series', 'cute_girls_doing_cute_things', 'romance_anime',
    'television_series_based_on_a_novel', 'urban_fantasy', 'post-apocalyptic_anime',
    'visual_novel', 'gag_anime', 'mystery_anime', 'supernatural_anime',
    'love_triangle_romance', 'post-apocalyptic_anime_and_manga', 'thriller_anime',
    'vampire_anime_and_manga', 'spokon', 'coming-of-age_anime_and_manga',
    'science_fiction_anime', 'yuri', 'comedy_drama_anime_and_manga',
    'historical_anime_and_manga', 'dark_fantasy', 'mystery_anime_and_manga',
    'horror_anime_and_manga', 'LGBT-related_television_series',
    'supernatural_anime_and_manga', 'ecchi', 'isekai', 'magical_girl', 'harem',
    'slice_of_life_anime_and_manga', 'romantic_comedy_anime_and_manga',
    'romance_anime_and_manga', 'school_anime_and_manga', 'drama_anime_and_manga', 'mecha',
    'adventure_anime_and_manga', 'science_fiction_anime_and_manga',
    'action_anime_and_manga', 'comedy_anime_and_manga', 'fantasy_anime_and_manga'
]

# 3. Definición de Categorías Maestras (Taxonomía)
# Diccionario: { "Nombre_De_Clase_Padre": [palabras_clave_para_detectarla] }
taxonomy = {
    "action_Genre": ["action", "fighting", "martial", "swashbuckler", "samurai", "ninja", "wuxia", "gun", "military", "police", "superhero"],
    "adventure_Genre": ["adventure", "isekai", "journey", "survival", "pirate", "western"],
    "comedy_Genre": ["comedy", "parody", "gag", "satire", "funny", "manzai"],
    "drama_Genre": ["drama", "tragedy", "melodrama", "legal", "medical"],
    "fantasy_Genre": ["fantasy", "magic", "supernatural", "mythology", "gods", "demon", "sorcery", "vampire", "werewolf", "ghost", "magical_girl"],
    "sciFi_Genre": ["science_fiction", "scifi", "cyberpunk", "steampunk", "mecha", "space", "robot", "time_travel", "dystopian", "apocalyptic", "biopunk", "tech"],
    "romance_Genre": ["romance", "romantic", "love", "dating", "harem", "bl", "yuri", "yaoi", "boys'_love"],
    "horror_Genre": ["horror", "terror", "zombie", "gore", "thriller", "suspense", "monster"],
    "sports_Genre": ["sport", "baseball", "football", "soccer", "basketball", "tennis", "fishing", "mahjong", "gambling", "spokon"],
    "sliceOfLife_Genre": ["slice_of_life", "school", "daily_life", "iyashikei", "cooking", "food", "fashion", "work_life", "educational", "coming-of-age"],
    "music_Genre": ["music", "idol", "band", "singer", "rhythm"],
    "history_Genre": ["historical", "history", "period", "jidaigeki", "alternate_history"],
    "adult_Themes": ["hentai", "ecchi", "eroge", "sex"],

    "Gourmet": ["gourmet", "food", "cooking", "sushi", "culture", "culinary"],
    "Erotica": ["erotica", "erotic"], 
    "Girls_Love": ["girls_love", "yuri", "shoujo_ai", "gl", "girls'_anime"],
    "Avant_Garde": ["avant_garde", "experimental", "art", "abstract", "psychological_anime"], # A veces lo experimental se solapa con lo psicológico
    "Slice_of_Life": ["slice_of_life", "daily_life", "iyashikei", "work_life", "coming-of-age", "fashion", "educational"],
    "Boys_Love": ["boys_love", "yaoi", "shounen_ai", "bl", "boys'_love"],
    "Suspense": ["suspense", "thriller", "psychological", "tech_noir"],
    "Award_Winning": ["award", "winning"],
    "Sports": ["sport", "baseball", "football", "soccer", "basketball", "tennis", "fishing", "mahjong", "gambling", "spokon", "association_football"],
    "Horror": ["horror", "terror", "zombie", "gore", "monster"],
    "Ecchi": ["ecchi", "sexy", "fan_service", "harem"], # Harem suele ir asociado a Ecchi o Romance
    "Mystery": ["mystery", "detective", "police", "crime", "procedural", "legal", "trial"],
    "Supernatural": ["supernatural", "ghost", "spirit", "demon", "gods", "vampire", "werewolf", "mythology", "yokai", "occult"],
    "Hentai": ["hentai", "eroge", "sex", "adult", "18+"],
    "Romance": ["romance", "romantic", "love", "dating", "otome"],
    "Fantasy": ["fantasy", "magic", "sorcery", "isekai", "sword_and_sorcery", "magical_girl", "high_fantasy", "dark_fantasy"],
    "Action": ["action", "fighting", "martial", "swashbuckler", "samurai", "ninja", "wuxia", "gun", "military", "superhero", "sentai", "tokusatsu"],
    "Drama": ["drama", "tragedy", "melodrama", "medical"],
    "Adventure": ["adventure", "journey", "survival", "pirate", "western"],
    "SciFi": ["science_fiction", "scifi", "cyberpunk", "steampunk", "mecha", "space", "robot", "time_travel", "dystopian", "apocalyptic", "biopunk", "tech"],
    "Comedy": ["comedy", "parody", "gag", "satire", "funny", "manzai", "cringe_comedy"],
    
    # Formatos y Orígenes
    "game_Adaptation": ["game", "rpg", "visual_novel", "sim", "simulation"],
    "literature_Adaptation": ["novel", "literature", "comics", "manga", "book"],
    "movie_Format": ["film", "movie", "cinema"],
    "tv_Format": ["television", "tv", "broadcast", "series", "program", "show"],
    "culture": ["japanese", "sushi", "national", "culture"]
}

# Crear las clases maestras en el grafo primero
for category in taxonomy.keys():
    cat_uri = ONTO[category]
    g.add((cat_uri, RDF.type, OWL.Class))
    g.add((cat_uri, RDFS.label, Literal(category.replace("_", " "))))
    g.add((cat_uri, RDFS.subClassOf, ONTO["AnimeSeries"])) # Todas heredan de AnimeSeries

# 4. Función Inteligente de Clasificación
def create_semantic_class(name_raw):
    # Limpieza básica
    name_clean = name_raw.replace("_manga", "").replace("/", "_").replace(" ", "_")
    
    # Nombre final de la clase (URI)
    if not name_clean.endswith("_anime"):
        class_name = f"{name_clean}_anime"
    else:
        class_name = name_clean
        
    class_name = class_name.lower()
    class_uri = ONTO[class_name]
    label = " ".join(word.capitalize() for word in name_clean.split("_")) + " (Genre)"
    
    # Añadir al grafo
    g.add((class_uri, RDF.type, OWL.Class))
    g.add((class_uri, RDFS.label, Literal(label)))
    
    # --- LÓGICA DE HERENCIA SEMÁNTICA ---
    parents_found = []
    name_lower = name_clean.lower()
    
    # Buscar coincidencias en la taxonomía
    for parent_class, keywords in taxonomy.items():
        for keyword in keywords:
            # Comprobar si la palabra clave está en el nombre (ej: "horror" en "comedy_horror")
            if keyword in name_lower:
                parents_found.append(ONTO[parent_class])
                break # Si ya encontramos que es "Action", no hace falta buscar más keywords de Action
    
    # Asignar padres encontrados
    if parents_found:
        for parent in parents_found:
            g.add((class_uri, RDFS.subClassOf, parent))
    else:
        # Si no encaja en ninguna categoría conocida, hereda directamente de la raíz
        g.add((class_uri, RDFS.subClassOf, ONTO["AnimeSeries"]))

# 5. Ejecutar
unique_genres = sorted(list(set(genres_uris)))
for genre in unique_genres:
    create_semantic_class(genre)

# 6. Guardar
output_file = "anime_ontology_semantic.ttl"
sbc.save(g, output_file, format="turtle")
print(f"Generado {output_file} con jerarquía semántica corregida.")


import rdflib
from rdflib import Graph, Namespace, RDF, RDFS, OWL, URIRef, Literal

# 1. Configuración de Namespaces
ONTO = Namespace("http://www.fdi.ucm.es/mia/sbc/animeontology/")
NS2 = Namespace("http://www.semanticweb.org/eladi/ontologies/anime-ontology#")
WD = Namespace("http://www.wikidata.org/entity/")

# 2. Cargar la Ontología Base (la estructura jerárquica)
g = Graph()
try:
    print("Cargando estructura de la ontología...")
    g = sbc.load("anime_ontology_semantic.ttl", format="turtle")
except FileNotFoundError:
    print("Error: No se encuentra 'anime_ontology_semantic.ttl'.")
    exit()

# 3. Cargar los Datos Nuevos (instancias)
data_g = Graph()
try:
    print("Cargando datos brutos...")
    data_g = sbc.load("expansionDeDominioInfinito.ttl", format="turtle")
except FileNotFoundError:
    print("Error: No se encuentra 'anime_data.ttl'.")
    exit()

# 4. Función de mapeo (URI sucia -> Clase Ontología)
def get_ontology_class(genre_resource):
    # Extraer nombre de la URI
    name_raw = genre_resource.split("/")[-1]
    # Limpieza idéntica a la creación de clases
    name_clean = name_raw.replace("anime_and_manga", "").replace("/", "_").replace(" ", "_").replace("_manga", "").replace("_anime", "")

    if not name_clean.endswith("_anime"):
        class_name = f"{name_clean}_anime"
    else:
        class_name = name_clean
    class_name = class_name.lower()
    return ONTO[class_name]

# --- PASO CRÍTICO: Pre-análisis ---
# Identificamos qué series tienen al menos un género asignado.
# Esto es necesario porque RDF no tiene orden, y necesitamos saber esto
# antes de decidir si borramos el tipo "AnimeSeries".
series_with_genre = set()

print("Analizando series con género...")
for s, p, o in data_g:
    if p == ONTO.hasGenre or p == NS2.hasGenre:
        series_with_genre.add(s)

# 5. Procesamiento y Fusión Inteligente
print("Fusionando grafos aplicando lógica de herencia...")

count_specialized = 0
count_generic = 0

for s, p, o in data_g:
    
    # CASO A: Es una relación de género (hasGenre)
    if p == ONTO.hasGenre or p == NS2.hasGenre:
        # En lugar de 'hasGenre', decimos que ES de ese tipo (rdf:type)
        target_class = get_ontology_class(o)
        g.add((s, RDF.type, target_class))
        count_specialized += 1
        
    # CASO B: Es la declaración de tipo genérico (rdf:type AnimeSeries)
    elif p == RDF.type and o == ONTO.AnimeSeries:
        # Solo añadimos que es "AnimeSeries" explícitamente si NO tiene ningún género específico.
        # Si ya tiene género, la ontología infiere que es un AnimeSeries, así que ahorramos el triple.
        if s not in series_with_genre:
            g.add((s, RDF.type, ONTO.AnimeSeries))
            count_generic += 1
        # Si está en series_with_genre, IGNORAMOS este triple.
            
    # CASO C: Cualquier otro dato (título, episodios, director, etc.)
    else:
        g.add((s, p, o))

# 6. Guardar
output_file = "end.ttl"
sbc.save(g, output_file, format="turtle")

print(f"--------------------------------------------------")
print(f"¡Grafo generado exitosamente: {output_file}")
print(f"Estadísticas:")
print(f" - Series clasificadas en géneros específicos: {len(series_with_genre)} (triples generados: {count_specialized})")
print(f" - Series sin género (se mantuvieron como AnimeSeries genérico): {count_generic}")
print(f"--------------------------------------------------")

Ontología guardada en: data\anime_ontology_semantic.ttl
Generado anime_ontology_semantic.ttl con jerarquía semántica corregida.
Cargando estructura de la ontología...
Cargando datos brutos...
Analizando series con género...
Fusionando grafos aplicando lógica de herencia...
Ontología guardada en: data\end.ttl
--------------------------------------------------
¡Grafo generado exitosamente: end.ttl
Estadísticas:
 - Series clasificadas en géneros específicos: 2712 (triples generados: 6966)
 - Series sin género (se mantuvieron como AnimeSeries genérico): 1363
--------------------------------------------------


In [11]:
# como vemos, han quedado algunas clases basura, producto de los cambios realizados, eliminanos "hasGenre" y todo lo que sea a onto:Genre
grafo = g
onto = Namespace("http://www.fdi.ucm.es/mia/sbc/animeontology/")
ns1 = Namespace("http://www.semanticweb.org/eladi/ontologies/anime-ontology#")

generos = set(grafo.subjects(RDF.type, onto.Genre))

print("Géneros encontrados:", len(generos))

for genre in generos:
    for p, o in list(grafo.predicate_objects(genre)):
        grafo.remove((genre, p, o))
    
    for s, p in list(grafo.subject_predicates(genre)):
        grafo.remove((s, p, genre))
    
    grafo.remove((genre, RDF.type, onto.Genre))

grafo.remove((onto.Genre, RDF.type, OWL.Class))

# eliminamos tambien la propiedad hasGenre
props = [
    onto.hasGenre,
    ns1.hasGenre
]

for prop in props:
    for p, o in list(grafo.predicate_objects(prop)):
        grafo.remove((prop, p, o))
    
    for s, p in list(grafo.subject_predicates(prop)):
        grafo.remove((s, p, prop))

#eliminanos la propiedad hasgenre de animeseries
grafo.remove((onto.hasGenre, RDF.type, OWL.ObjectProperty))
sbc.print_class_hierarchy(grafo)

sbc.save(grafo, "ontologiaAnimes.ttl")



Géneros encontrados: 21

JERARQUÍA DE CLASES DE LA ONTOLOGÍA
📁 Girls' Anime (Genre) (girls'_anime)
📁 Production Role (anime-ontology#ProductionRole) [anime-ontology#productionRoleIn]
  └─ Art_Director: Sin etiqueta
  └─ 2nd_Key_Animation: Sin etiqueta
  └─ Music: Sin etiqueta
  └─ Animation_Check: Sin etiqueta
  └─ Storyboard: Sin etiqueta
  └─ Chief_Animation_Director: Sin etiqueta
  └─ Character_Design: Sin etiqueta
  └─ Planning_Producer: Sin etiqueta
  └─ Editing: Sin etiqueta
  └─ Assistant_Producer: Sin etiqueta
  └─ Director_of_Photography: Sin etiqueta
  └─ Publicity: Sin etiqueta
  └─ Layout: Sin etiqueta
  └─ Executive_Producer: Sin etiqueta
  └─ Online_Editor: Sin etiqueta
  └─ Setting_Manager: Sin etiqueta
  └─ Theme_Song_Arrangement: Sin etiqueta
  └─ Inserted_Song_Performance: Sin etiqueta
  └─ Chief_Producer: Sin etiqueta
  └─ Recording_Engineer: Sin etiqueta
  └─ Director: Sin etiqueta
  └─ Production_Assistant: Sin etiqueta
  └─ Production_Manager: Sin etiqueta
  └─ Me

In [17]:
import numpy as np
from typing import List, Tuple, Dict
import torch
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from rdflib import Graph, Namespace, RDF, RDFS, Literal
import sbc_tools as sbc

def entrenar_transe_own(triples: List[Tuple[str, str, str]], 
                    embedding_dim: int = 128,  # Increased dim
                    num_epochs: int = 300):    # Increased epochs
    
    print("=" * 70)
    print("PRE-PROCESSING DATA WEIGHTS")
    
    # 1. OVERSAMPLING GENRES
    final_triples = []
    for h, r, t in triples:
        final_triples.append((h, r, t))
        # Check for RDF type or Genre relations
        if 'type' in r or 'Genre' in t or 'genre' in r:
            # Replicate genre triples 5 to 10 times
            for _ in range(10): 
                final_triples.append((h, r, t))
                
    print(f"Triples original: {len(triples)} -> Weighted: {len(final_triples)}")

    # 2. Create Factory
    tf = TriplesFactory.from_labeled_triples(
        np.array(final_triples, dtype=str),
        create_inverse_triples=True
    )
    
    # 3. Train with Bernoulli Sampling
    result = pipeline(
        model='TransE',
        training=tf,
        testing=tf,
        model_kwargs={
            'embedding_dim': embedding_dim,
            'scoring_fct_norm': 1  # L1 norm often works better for sparse graphs than L2
        },
        optimizer='Adam',
        optimizer_kwargs={'lr': 0.001}, # Slightly lower LR for more epochs
        
        # IMPORTANT: Handle 1-to-N relations better
        negative_sampler='bernoulli',
        
        training_kwargs={
            'num_epochs': num_epochs,
            'batch_size': 64
        },
        random_seed=42,
        device='gpu'
    )
    
    return result


# ============================================================================
# PARTE 4: CÁLCULO DE SIMILITUD ENTRE PELÍCULAS
# ============================================================================

def obtener_embedding_entidad(model, entity_name: str, entity_to_id: Dict) -> np.ndarray:
    """
    Obtiene el vector embedding de una entidad
    """
    if entity_name not in entity_to_id:
        raise ValueError(f"Entidad '{entity_name}' no encontrada en el grafo")
    
    entity_id = entity_to_id[entity_name]
    embedding = model.entity_representations[0](
        torch.tensor([entity_id])
    ).detach().numpy()[0]
    
    return embedding

def calcular_similitudes_peliculas(result, peliculas: List[str]):
    """
    Calcula la matriz de similitud entre películas usando embeddings TransE
    """
    model = result.model
    entity_to_id = result.training.entity_to_id
    
    print("\n" + "=" * 70)
    print("CÁLCULO DE SIMILITUDES CON EMBEDDINGS TransE")
    print("=" * 70)
    
    # Obtener embeddings de todas las películas
    embeddings = []
    peliculas_validas = []
    
    for pelicula in peliculas:
        try:
            emb = obtener_embedding_entidad(model, pelicula, entity_to_id)
            embeddings.append(emb)
            peliculas_validas.append(pelicula)
        except ValueError as e:
            print(f"⚠ {e}")
    
    embeddings = np.array(embeddings)
    
    # Calcular matriz de similitud coseno
    sim_matrix = cosine_similarity(embeddings)
    
    # Mostrar resultados
    print(f"\nMatriz de similitud (coseno) entre {len(peliculas_validas)} películas:\n")
    print(f"{'':20}", end='')
    for p in peliculas_validas:
        print(f"{p:12}", end='')
    print()
    
    for i, p1 in enumerate(peliculas_validas):
        print(f"{p1:20}", end='')
        for j, p2 in enumerate(peliculas_validas):
            print(f"{sim_matrix[i][j]:12.4f}", end='')
        print()
    
    return sim_matrix, peliculas_validas, embeddings

def encontrar_peliculas_similares(result, pelicula_query: str, top_k: int = 3):
    """
    Encuentra las k películas más similares a una película dada
    """
    model = result.model
    entity_to_id = result.training.entity_to_id
    id_to_entity = {v: k for k, v in entity_to_id.items()}
    
    # Obtener todas las películas del grafo
    todas_peliculas = [ent for ent in entity_to_id.keys() 
                       if ent not in ['type', 'Movie', 'director', 'actor', 'genre', 
                                     'name', 'label', 'SciFi', 'Drama', 'Action', 
                                     'Thriller', 'Romance', 'Crime']]
    
    # Obtener embedding de la película query
    try:
        query_emb = obtener_embedding_entidad(model, pelicula_query, entity_to_id)
    except ValueError as e:
        print(f"Error: {e}")
        return
    
    # Calcular similitud con todas las películas
    similitudes = []
    for pelicula in todas_peliculas:
        if pelicula != pelicula_query:
            try:
                emb = obtener_embedding_entidad(model, pelicula, entity_to_id)
                sim = cosine_similarity([query_emb], [emb])[0][0]
                similitudes.append((pelicula, sim))
            except:
                pass
    
    # Ordenar por similitud
    similitudes.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n{'=' * 70}")
    print(f"Top {top_k} películas similares a '{pelicula_query}':")
    print(f"{'=' * 70}")
    for i, (pelicula, sim) in enumerate(similitudes[:top_k], 1):
        print(f"{i}. {pelicula:20} (similitud: {sim:.4f})")

def rdf_a_triples(g: Graph) -> List[Tuple[str, str, str]]:
    """
    Convierte un grafo RDF a formato de triples para PyKEEN
    Formato: (sujeto, predicado, objeto)
    """
    triples = []
    for s, p, o in g:
        # Convertir URIs y literales a strings simplificados
        subj = str(s).split('#')[-1] if '#' in str(s) else str(s).split('/')[-1]
        pred = str(p).split('#')[-1] if '#' in str(p) else str(p).split('/')[-1]
        obj = str(o).split('#')[-1] if '#' in str(o) else str(o).split('/')[-1]
        
        # Filtrar literales de nombres (opcional, para simplificar)
        if not obj.startswith('"'):
            triples.append((subj, pred, obj))
    
    return triples

In [15]:
g = sbc.load("end.ttl")

tripletas = rdf_a_triples(g)

In [ ]:
model = entrenar_transe_own(tripletas, embedding_dim=50, num_epochs=100)

from pykeen.checkpoints import save_model
model.save_to_directory("graph_embeddings_project")

PRE-PROCESSING DATA WEIGHTS
Triples original: 373611 -> Weighted: 1379261


INFO:pykeen.pipeline.api:Using device: gpu
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.triples.triples_factory:Creating inverse triples.
Training epochs on cpu:   1%|          | 1/100 [10:27<17:15:11, 627.39s/epoch, loss=0.278, prev_loss=nan]